# 📖 Lab 4: Resumable Uploads (Deep Dive)

**Non-functional requirement:** *The system should support resumable uploads for large videos (10s of GBs).*

A 10GB video upload over unstable WiFi can fail at any point. Without resumability, the user loses all progress and must restart from scratch. AWS S3 / MinIO support **multipart upload** natively — we just need to track chunk progress in our metadata.

## 🏗️ Architecture — Before (Lab 1: Single Upload)

```
┌────────┐  PUT entire file via presigned URL  ┌──────────┐
│ Client │────────────────────────────────────>│   S3     │
└────────┘     ❌ network fails at 80%         └──────────┘
               → lose ALL progress, start over
```

## 🏗️ Architecture — After (Multipart + Chunk Tracking)

```
┌────────┐  POST /presigned_url  ┌─────────────┐  chunks[]  ┌──────────┐
│        │─────────────────────>│Upload Service│──────────>│Metadata  │
│        │<────{chunkURLs[]}── │             │           │DB        │
│ Client │                       └─────────────┘           │[{hash,   │
│        │  PUT chunk 1 ────────────────────────> S3        │ status}] │
│        │  PUT chunk 2 ────────────────────────> S3        └──────────┘
│        │  ❌ fails at chunk 3
│        │  ...reconnect...
│        │  GET /chunks → [1=✅, 2=✅, 3=❌, 4=❌, 5=❌]
│        │  PUT chunk 3 ────────────────────────> S3  ← resume!
│        │  PUT chunk 4, 5 ─────────────────────> S3
│        │  CompleteMultipartUpload ────> S3 → processing pipeline
└────────┘
```

## Learning Objectives

- Understand S3/MinIO multipart upload API
- Split a file into chunks with fingerprint hashes
- Track chunk upload progress in the metadata DB
- Simulate a network failure and resume from where we left off
- See the complete flow: init → upload chunks → verify → complete

## 🛠️ Setup

```bash
cd system-designs/youtube
docker-compose up -d
```

Select the **"YouTube (Python)"** kernel.

In [ ]:
import psycopg2
import psycopg2.extras
from minio import Minio
import hashlib
import uuid
import time
import os
import io
import json

DB_CONFIG = {
    "host": "localhost", "port": 5434,
    "user": "demo", "password": "demo", "database": "youtube",
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

minio_client = Minio("localhost:9000", access_key="minioadmin", secret_key="minioadmin", secure=False)

# Add chunks column to videos table if not exists
conn = get_connection()
cur = conn.cursor()
cur.execute("""
    DO $$ BEGIN
        ALTER TABLE videos ADD COLUMN chunks JSONB;
    EXCEPTION WHEN duplicate_column THEN NULL;
    END $$;
""")
conn.commit()
cur.close()
conn.close()

print(f"✅ PostgreSQL: connected")
print(f"✅ MinIO: {[b.name for b in minio_client.list_buckets()]}")

## 📊 How Multipart Upload Works

S3/MinIO multipart upload has 3 steps:

```
1. CreateMultipartUpload  → returns an uploadId
2. UploadPart (repeat)    → upload each chunk, get back ETag per part
3. CompleteMultipartUpload → S3 assembles all parts into one object
```

If the upload fails, we can **list uploaded parts** and resume from where we left off. S3 keeps the partial upload alive until we complete or abort it.

We'll add our own **chunk tracking** in the metadata DB so the server knows the status of each chunk independently.

## Step 1: Client Splits File into Chunks

The client divides the video into fixed-size chunks (~5MB each) and computes a fingerprint hash for each. This happens entirely on the client side.

In [ ]:
CHUNK_SIZE = 5 * 1024 * 1024  # 5 MB per chunk

# Create a simulated 25MB video file
FILE_SIZE = 25 * 1024 * 1024
fake_video = os.urandom(FILE_SIZE)

def split_into_chunks(file_data: bytes, chunk_size: int) -> list[dict]:
    """Client-side: split file into chunks with fingerprint hashes."""
    chunks = []
    offset = 0
    part_num = 1

    while offset < len(file_data):
        chunk_data = file_data[offset : offset + chunk_size]
        fingerprint = hashlib.md5(chunk_data).hexdigest()
        chunks.append({
            "partNumber": part_num,
            "offset": offset,
            "size": len(chunk_data),
            "fingerprint": fingerprint,
            "status": "NotUploaded",
        })
        offset += chunk_size
        part_num += 1

    return chunks


chunks = split_into_chunks(fake_video, CHUNK_SIZE)

print(f"🎬 File: {FILE_SIZE / 1024 / 1024:.0f}MB split into {len(chunks)} chunks of ~{CHUNK_SIZE / 1024 / 1024:.0f}MB\n")
print(f"  {'Part':<6} {'Size':<10} {'Fingerprint':<36} {'Status'}")
print(f"  {'─'*65}")
for c in chunks:
    print(f"  {c['partNumber']:<6} {c['size'] // 1024}KB    {c['fingerprint']:<36} {c['status']}")

## Step 2: Initialize Multipart Upload

Client calls `POST /presigned_url` with chunk info. Server creates metadata with `chunks[]` and initiates a multipart upload in S3.

In [ ]:
from minio.helpers import MIN_PART_SIZE
from urllib.parse import urljoin

VIDEO_ID = uuid.uuid4().hex[:12]
OBJECT_NAME = f"{VIDEO_ID}/original.mp4"
BUCKET = "raw-videos"

# Server side: create metadata with chunk tracking
conn = get_connection()
cur = conn.cursor()
cur.execute("""
    INSERT INTO videos (id, title, user_id, status, content_type, file_size_bytes, raw_url, chunks)
    VALUES (%s, 'Resumable Upload Demo', 1, 'uploading', 'video/mp4', %s, %s, %s)
""", (VIDEO_ID, FILE_SIZE, f"{BUCKET}/{OBJECT_NAME}", json.dumps(chunks)))
conn.commit()
cur.close()
conn.close()

# Server side: initiate multipart upload in S3
upload_id = minio_client._create_multipart_upload(BUCKET, OBJECT_NAME, headers={})

print(f"📌 Multipart upload initiated:")
print(f"   videoId: {VIDEO_ID}")
print(f"   uploadId: {upload_id}")
print(f"   chunks: {len(chunks)}")
print(f"\n   💡 S3 now knows we're going to upload {len(chunks)} parts.")
print(f"      Each part can be uploaded independently.")

## Step 3: Upload Chunks (with simulated failure)

We'll upload chunks 1 and 2 successfully, then simulate a network failure on chunk 3. This leaves the upload **partially complete**.

In [ ]:
uploaded_parts = []  # Tracks {partNumber, etag} for CompleteMultipartUpload
FAIL_AT_CHUNK = 3    # Simulate failure on chunk 3

def upload_chunk(part_number: int, chunk_data: bytes) -> dict:
    """Upload one chunk to S3 as a multipart part."""
    etag = minio_client._upload_part(
        BUCKET, OBJECT_NAME, upload_id,
        part_number, chunk_data, len(chunk_data), None, None,
    )
    return {"partNumber": part_number, "etag": etag}


def update_chunk_status(video_id: str, part_number: int, status: str, etag: str = None):
    """Update chunk status in metadata DB."""
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT chunks FROM videos WHERE id = %s", (video_id,))
    chunk_list = cur.fetchone()[0]

    for c in chunk_list:
        if c["partNumber"] == part_number:
            c["status"] = status
            if etag:
                c["etag"] = etag
            break

    cur.execute("UPDATE videos SET chunks = %s WHERE id = %s", (json.dumps(chunk_list), video_id))
    conn.commit()
    cur.close()
    conn.close()


print(f"📤 Uploading chunks (will fail at chunk {FAIL_AT_CHUNK}):\n")

for i, chunk_info in enumerate(chunks):
    part_num = chunk_info["partNumber"]
    offset = chunk_info["offset"]
    size = chunk_info["size"]
    chunk_data = fake_video[offset : offset + size]

    if part_num == FAIL_AT_CHUNK:
        print(f"  ❌ Chunk {part_num}: NETWORK FAILURE! Upload interrupted.")
        print(f"\n  😡 Lost connection. {part_num - 1} of {len(chunks)} chunks uploaded.")
        break

    result = upload_chunk(part_num, chunk_data)
    uploaded_parts.append(result)
    update_chunk_status(VIDEO_ID, part_num, "Uploaded", result["etag"])
    print(f"  ✅ Chunk {part_num}: uploaded ({size // 1024}KB), etag={result['etag'][:16]}...")

# Show current state
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT chunks FROM videos WHERE id = %s", (VIDEO_ID,))
current_chunks = cur.fetchone()["chunks"]
cur.close()
conn.close()

print(f"\n📊 Chunk status in metadata DB:")
for c in current_chunks:
    icon = "✅" if c["status"] == "Uploaded" else "⏳"
    print(f"  {icon} Part {c['partNumber']}: {c['status']}")

## Step 4: Resume Upload

The client reconnects. It fetches the chunk status from the server and **only uploads the missing chunks**. No data is re-uploaded.

In [ ]:
# Client reconnects and fetches chunk status: GET /videos/:id/chunks
print("🔄 Client reconnects. Fetching chunk status...\n")

conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT chunks FROM videos WHERE id = %s", (VIDEO_ID,))
saved_chunks = cur.fetchone()["chunks"]
cur.close()
conn.close()

# Determine which chunks need uploading
not_uploaded = [c for c in saved_chunks if c["status"] == "NotUploaded"]
already_uploaded = [c for c in saved_chunks if c["status"] == "Uploaded"]

print(f"  Already uploaded: {len(already_uploaded)} chunks (skipping)")
print(f"  Need to upload:   {len(not_uploaded)} chunks\n")

# Resume: upload only the missing chunks
print("📤 Resuming upload:\n")

for chunk_info in not_uploaded:
    part_num = chunk_info["partNumber"]
    offset = chunk_info["offset"]
    size = chunk_info["size"]
    chunk_data = fake_video[offset : offset + size]

    result = upload_chunk(part_num, chunk_data)
    uploaded_parts.append(result)
    update_chunk_status(VIDEO_ID, part_num, "Uploaded", result["etag"])
    print(f"  ✅ Chunk {part_num}: uploaded ({size // 1024}KB) — RESUMED")

print(f"\n🎉 All {len(chunks)} chunks uploaded! No data was re-uploaded.")

## Step 5: Complete Multipart Upload

All parts are uploaded. Client calls `CompleteMultipartUpload` — S3 assembles all parts into a single object. This triggers the processing pipeline.

In [ ]:
# Sort parts by part number (S3 requires them in order)
uploaded_parts.sort(key=lambda x: x["partNumber"])

print("📌 Completing multipart upload...\n")
print(f"  Parts to assemble: {len(uploaded_parts)}")
for p in uploaded_parts:
    print(f"    Part {p['partNumber']}: etag={p['etag'][:16]}...")

# Complete the multipart upload — S3 assembles the object
minio_client._complete_multipart_upload(BUCKET, OBJECT_NAME, upload_id, uploaded_parts)

# Verify the assembled object
stat = minio_client.stat_object(BUCKET, OBJECT_NAME)
print(f"\n  ✅ Multipart upload complete!")
print(f"     Object: {stat.object_name}")
print(f"     Size: {stat.size:,} bytes ({stat.size / 1024 / 1024:.1f}MB)")
print(f"     Original file: {FILE_SIZE:,} bytes")
print(f"     Match: {'✅ Yes!' if stat.size == FILE_SIZE else '❌ No'}")

# Update metadata status
conn = get_connection()
cur = conn.cursor()
cur.execute("UPDATE videos SET status = 'uploaded' WHERE id = %s", (VIDEO_ID,))
conn.commit()
cur.close()
conn.close()

print(f"\n  📊 Video {VIDEO_ID} status: uploaded → ready for processing pipeline")
print(f"     In production, S3 would emit ObjectCreated:CompleteMultipartUpload event")

## 🧹 Cleanup

In [ ]:
conn = get_connection()
cur = conn.cursor()
cur.execute("DELETE FROM videos WHERE id = %s", (VIDEO_ID,))
conn.commit()
cur.close()
conn.close()

minio_client.remove_object(BUCKET, OBJECT_NAME)
print("✅ Cleaned up test data.")

## ✅ Summary

### The Resumable Upload Flow

```
Client                          Server                          S3
──────                          ──────                          ──
1. Split file into chunks
   + compute fingerprints
                                2. Create VideoMetadata
                                   with chunks[] status
                                                                3. CreateMultipartUpload
                                                                   → uploadId
4. Upload chunk 1 ──────────────────────────────────────────>  UploadPart(1)  → etag
   notify server ──────────────> mark chunk 1 = Uploaded
5. Upload chunk 2 ──────────────────────────────────────────>  UploadPart(2)  → etag
   notify server ──────────────> mark chunk 2 = Uploaded

   ❌ NETWORK FAILURE

   ...reconnect...

6. Fetch chunk status <─────── GET /videos/:id/chunks
   [1=✅, 2=✅, 3=⏳, 4=⏳, 5=⏳]

7. Skip 1,2. Upload 3,4,5 ─────────────────────────────────> UploadPart(3,4,5)
   notify server each ─────────> mark 3,4,5 = Uploaded

8. CompleteMultipartUpload ─────────────────────────────────> Assemble → event
                                                               → processing pipeline
```

### Key Design Decisions

| Decision | Why |
|----------|-----|
| **Chunk fingerprint (MD5)** | Verify data integrity — detect corruption during upload |
| **Server-side chunk tracking** | Client can disconnect; chunk status persists in DB for resume |
| **S3 multipart upload** | Native support — handles assembly, concurrency, and cleanup |
| **5-10MB chunks** | Small enough to retry quickly on failure, large enough to minimize overhead |
| **Client-driven progress** | Client decides when to upload, server just tracks. Stateless server. |

### Production Considerations

| Concern | Solution |
|---------|----------|
| **Abandoned uploads** | S3 lifecycle rules to abort multiparts older than 24h |
| **Concurrent chunk uploads** | Client can upload multiple chunks in parallel for speed |
| **Integrity verification** | Compare chunk fingerprints with S3 ETags server-side |
| **Progress UI** | `(uploaded_chunks / total_chunks) × 100` = progress bar percentage |

**Pattern reference:** See `patterns/large-blobs/` for a deeper exploration of resumable uploads, content-addressable storage, and chunked transfer.